# V4 Universal Football Model — Footballdata.io Ingestion

This notebook tests the new Footballdata.io API, replacing the deprecated Sofascore integration.
We will use this notebook to explore the JSON structure and build the parsing logic before wiring it into the live V4 backend.

In [1]:
import os
import json
import requests
from dotenv import load_dotenv

# Load the API key from .env
load_dotenv(dotenv_path="../.env")
API_KEY = os.getenv("FOOTBALLDATA_API_KEY")

if not API_KEY:
    print("⚠️ API Key not found! Please check your .env file.")
else:
    print(f"✅ API Key loaded: {API_KEY[:5]}...{API_KEY[-5:]}")

✅ API Key loaded: fd_75...6542e


## 1. Basic API Fetch Function
Let's define a helper function to hit the Footballdata.io endpoints based on their documentation.

In [2]:
# The correct base URL according to https://footballdata.io/documentation/endpoints/
BASE_URL = "https://footballdata.io/api/v1"

def fetch_footballdata(endpoint, params=None):
    """Helper to fetch data from footballdata.io"""
    if params is None:
        params = {}
        
    url = f"{BASE_URL}/{endpoint.lstrip('/')}"
    
    # Standard Authorization approaches.
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Accept": "application/json"
    }
    
    try:
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status() # Raise an exception for bad status codes
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"❌ Error fetching data: {e}")
        if 'response' in locals() and response is not None:
            print(f"Response text: {response.text}")
        return None

## 2. Test Connection
Let's do a basic test to make sure the key works and we can view the data payload structure.

In [3]:
# Fetching a list of active leagues
test_endpoint = "leagues"

print(f"Fetching endpoint: /{test_endpoint}...")
data = fetch_footballdata(test_endpoint)

if data:
    print("\n✅ Success! Here is a snippet of the response:\n")
    print(json.dumps(data, indent=2)[:1000] + "\n...[truncated]")
else:
    print("\n⚠️ Failed to retrieve data. Check endpoint path or auth headers.")

Fetching endpoint: /leagues...

✅ Success! Here is a snippet of the response:

{
  "success": true,
  "data": [
    {
      "league_id": 15,
      "league_name": "Premier League",
      "country": "England",
      "league_image": "https://footballdata.io/img/league/england-premier-league.png",
      "seasons_available": 20,
      "earliest_season": 20072008,
      "latest_season": 20262027
    },
    {
      "league_id": 45,
      "league_name": "UEFA Champions League",
      "country": "Europe",
      "league_image": "https://footballdata.io/img/league/europe-uefa-champions-league.png",
      "seasons_available": 18,
      "earliest_season": 20092010,
      "latest_season": 20262027
    },
    {
      "league_id": 46,
      "league_name": "UEFA Europa League",
      "country": "Europe",
      "league_image": "https://footballdata.io/img/league/europe-uefa-europa-league.png",
      "seasons_available": 18,
      "earliest_season": 20092010,
      "latest_season": 20262027
    },
    {
